### U-Net architecture for the radio maps estimation and cells clasification, for indoor scennarios

#### Load some libraries

In [ ]:
import tensorflow as tf 
from tensorflow.keras import optimizers
from tensorflow import keras
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, ReLU
from tensorflow.keras.layers import BatchNormalization, Conv2DTranspose, Concatenate
from tensorflow.keras.models import Model, load_model
import numpy as np
import os
from skimage import io
import matplotlib.pyplot as plt
from tqdm import tqdm 
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import time
import random
import tensorflow.keras.backend as K
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import sys
# sys.path.append(os.path.abspath('E:/DataSet5GHz/Codes/Python files'))
# from normalize_map import normalize_map
from sklearn.utils import class_weight
from sklearn.preprocessing import MinMaxScaler
import matplotlib.cm as cm
from sklearn.metrics import f1_score
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.font_manager import FontProperties

#### Initialization of some generals parameters

In [ ]:
n_APs = 5
simulation = 'cells'
porc_test = 0.1
epochs = 500
batch_size = 16
learning_rate = 1e-3
higth_img = 256 
width_img = 256
scenarios = 60
positions = 150 # Choice amount of simulations for plain, maximum 1000.

# Dataset path

word_path = 'E'
dir_dataset = word_path + ':/DataSet5GHz/'

dir_dataset_esc = dir_dataset + 'Scennarios init/Scennarios B/'
if n_APs == 5:
    dir_dataset_txs = dir_dataset + 'Txs/5AP/'
    dir_cells = dir_dataset + 'Maps and cells/5AP/Cells/'
    dir_maps = dir_dataset + 'Maps and cells/5AP/Maps/'
if n_APs == 4:
    dir_dataset_txs = dir_dataset + 'Txs/4AP/'
    dir_cells = dir_dataset + 'Maps and cells/4AP/Cells/'
    dir_maps = dir_dataset + 'Maps and cells/4AP/Maps/'
if n_APs == 3:
    dir_dataset_txs = dir_dataset + 'Txs/3AP/'
    dir_cells = dir_dataset + 'Maps and cells/3AP/Cells/'
    dir_maps = dir_dataset + 'Maps and cells/3AP/Maps/'
if n_APs == 2:
    dir_dataset_txs = dir_dataset + 'Txs/2AP/'
    dir_cells = dir_dataset + 'Maps and cells/2AP/Cells/'
    dir_maps = dir_dataset + 'Maps and cells/2AP/Maps/'
if n_APs == 1:
    dir_dataset_txs = dir_dataset + 'Txs/1AP/'
    dir_cells = ''
    dir_maps = dir_dataset + 'Maps and cells/1AP/Maps/'

#### Functions architecture U-Net

In [ ]:
################################################################
def block_1convolution(input_b1, filters, kernel, train = True):
    conv1 = Conv2D(filters, kernel, padding = 'same', kernel_initializer='he_normal', trainable = train)(input_b1)
    batch_norm1 = BatchNormalization(trainable = train)(conv1)
    act1 = ReLU(trainable = train)(batch_norm1)
    
    return act1
################################################################
def block_2convolution(input_b2, filters, kernel, train = True):
    conv1 = Conv2D(filters, kernel, padding = 'same', kernel_initializer='he_normal', trainable = train)(input_b2)
    batch_norm1 = BatchNormalization(trainable = train)(conv1)
    act1 = ReLU(trainable = train)(batch_norm1)
    
    conv2 = Conv2D(filters, kernel, padding = 'same', kernel_initializer='he_normal', trainable = train)(act1)
    batch_norm2 = BatchNormalization(trainable = train)(conv2)
    act2 = ReLU(trainable = train)(batch_norm2)
    
    return act2
###############################################################
def encoder(input_enc, filters, kernel, train = True):
    enc1 = block_2convolution(input_enc, filters, kernel, train = train)
    MaxPool1 = MaxPooling2D(strides = (2,2), trainable = train)(enc1)
    return enc1, MaxPool1
###############################################################
def decoder(input_dec, skip, filters, kernel, train = True):
    Upsample = Conv2DTranspose(filters, (2, 2), strides=2, padding = 'same', trainable = train)(input_dec)
    Connect_Skip = Concatenate(trainable = train)([Upsample, skip])
    out = block_2convolution(Connect_Skip, filters, kernel, train = train)
    return out
###############################################################
def U_Net(n_APs, higth_img = higth_img, width_img = width_img, channels_img = None, simulation = ''):

    # Input
    input1 = Input((higth_img, width_img, channels_img))
    
    #256,256,deep
    if n_APs == 4:
        skip1_1, encoder_1_1 = encoder(input1, filters = channels_img + 20, kernel = 3) #128,128,deep
    if n_APs == 5:
        skip1_1, encoder_1_1 = encoder(input1, filters = channels_img + 30, kernel = 3) #128,128,deep
    if n_APs < 4:
        skip1_1, encoder_1_1 = encoder(input1, filters = channels_img + 6, kernel = 3) #128,128,deep
    skip2_1, encoder_2_1 = encoder(encoder_1_1, filters = 40, kernel = 3) #64,64,deep
    skip3_1, encoder_3_1 = encoder(encoder_2_1, filters = 50, kernel = 3) #32,32,deep
    skip4_1, encoder_4_1 = encoder(encoder_3_1, filters = 60, kernel = 3) #16,16,deep
    skip5_1, encoder_5_1 = encoder(encoder_4_1, filters = 100, kernel = 3) #8,8,deep
    skip6_1, encoder_6_1 = encoder(encoder_5_1, filters = 200, kernel = 3) #4,4,deep
    skip7_1, encoder_7_1 = encoder(encoder_6_1, filters = 300, kernel = 3) #2,2,deep
    skip8_1, encoder_8_1 = encoder(encoder_7_1, filters = 400, kernel = 3) #1,1,deep
    
    conv_block_1 = block_2convolution(encoder_8_1, filters = 600, kernel = 3) #1,1,deep
    
    decoder_1_1 = decoder(conv_block_1, skip8_1, filters = 400, kernel = 3) #2,2,deep
    decoder_2_1 = decoder(decoder_1_1, skip7_1, filters = 300, kernel = 3) #4,4,deep
    decoder_3_1 = decoder(decoder_2_1, skip6_1, filters = 200, kernel = 3) #8,8,deep
    decoder_4_1 = decoder(decoder_3_1, skip5_1, filters = 100, kernel = 3) #16,16,deep
    decoder_5_1 = decoder(decoder_4_1, skip4_1, filters = 60, kernel = 3) #32,32,deep
    decoder_6_1 = decoder(decoder_5_1, skip3_1, filters = 50, kernel = 3) #64,64,deep
    decoder_7_1 = decoder(decoder_6_1, skip2_1, filters = 40, kernel = 3) #128,128,deep       
    decoder_8_1 = decoder(decoder_7_1, skip1_1, filters = 20, kernel = 3) #256,256,deep
    
    conc1_1 = Concatenate()([decoder_8_1, input1])
    
    aditional_ = block_1convolution(conc1_1, filters = 20, kernel = 3) #256,256,deep
  
    conc2_1 = Concatenate()([aditional_, input1])
    
    if simulation == 'maps':
        out_1 = Conv2D(1, 1, padding = 'same', activation = 'relu', kernel_initializer='he_normal')(conc2_1)
    if simulation == 'cells':
        out_1 = Conv2D(n_APs, 1, padding = 'same', activation = 'softmax', kernel_initializer='he_normal')(conc2_1)
    
    model = Model(input1, out_1)
    
    return model
###############################################################

#### Architecture visualization

In [ ]:
if simulation == 'cells':
    channels_img = n_APs + 2
    # channels_img = 1
if simulation == 'maps':
    channels_img = n_APs + 1

model = U_Net(n_APs = n_APs, channels_img = channels_img, simulation = simulation)

model.summary()

"""tf.keras.utils.plot_model(model, 'C:/Users/GIIEE/Documents/johan/proyecto/modelos/model.png', 
                          show_shapes=False, show_layer_names=True, 
                          rankdir='TB', expand_nested=False, dpi=110)"""

#### Load data

In [ ]:
def data(maps_inds, simulation = '', n_APs = None, dir_dataset_txs = '', dir_cells = '', dir_maps = '',
        dir_dataset_esc = '', positions = None, ini = None, end = None, phase = '', scenarios = scenarios):
    
    if phase == 'test':
        maps_inds = maps_inds
    else:
        maps_inds = maps_inds[ini:end]

    images_input = []

    images_out = [] 

    for idx in tqdm(maps_inds):
        idxr = np.floor((idx - 1)/positions).astype(int)
        idxc = idx - (idxr * positions)
        if phase == 'test':
            dataset_map_ind = idxr + 1 + scenarios            
        else: 
            dataset_map_ind = idxr + 1 

        name_buildings = str(dataset_map_ind) + ".png"
        name_maps = str(dataset_map_ind) + "_" + str(idxc) + ".png"
        name_tx1 = str(dataset_map_ind) + "_" + str(idxc) + ".png"
        
        if n_APs > 1:
            name_tx1 = str(dataset_map_ind) + "_" + str(idxc) + "_" + '1' + ".png"
            name_tx2 = str(dataset_map_ind) + "_" + str(idxc) + "_" + '2' + ".png"
            
        if n_APs > 2:
            name_tx3 = str(dataset_map_ind) + "_" + str(idxc) + "_" + '3' + ".png"
            
        if n_APs > 3:
            name_tx4 = str(dataset_map_ind) + "_" + str(idxc) + "_" + '4' + ".png"
            
        if n_APs > 4:
            name_tx5 = str(dataset_map_ind) + "_" + str(idxc) + "_" + '5' + ".png"

        img_name_buildings = os.path.join(dir_dataset_esc, name_buildings)
        image_buildings = np.asarray(io.imread(img_name_buildings))/255

        img_name_Tx1 = os.path.join(dir_dataset_txs, name_tx1)
        image_Tx1 = np.asarray(io.imread(img_name_Tx1))/255

        if n_APs > 1:
            img_name_Tx2 = os.path.join(dir_dataset_txs, name_tx2)
            image_Tx2 = np.asarray(io.imread(img_name_Tx2))/255

        if n_APs > 2:
            img_name_Tx3 = os.path.join(dir_dataset_txs, name_tx3)
            image_Tx3 = np.asarray(io.imread(img_name_Tx3))/255
            
        if n_APs > 3:
            img_name_Tx4 = os.path.join(dir_dataset_txs, name_tx4)
            image_Tx4 = np.asarray(io.imread(img_name_Tx4))/255
            
        if n_APs > 4:
            img_name_Tx5 = os.path.join(dir_dataset_txs, name_tx5)
            image_Tx5 = np.asarray(io.imread(img_name_Tx5))/255

        if n_APs == 1:
            img = np.stack([image_buildings, image_Tx1], axis = 2)  
            images_input.append(img)    
            
        if n_APs == 2 and simulation == 'maps':
            img = np.stack([image_buildings, image_Tx1, image_Tx2], axis = 2)  
            images_input.append(img)
            
        if n_APs == 2 and simulation == 'cells':
            maps_p = os.path.join(dir_maps, name_maps)
            maps_p = np.asarray(io.imread(maps_p))/255
            img = np.stack([image_buildings, image_Tx1, image_Tx2, maps_p], axis = 2) 
            images_input.append(img)
            
        if n_APs == 3 and simulation == 'maps':
            img = np.stack([image_buildings, image_Tx1, image_Tx2, image_Tx3], axis = 2)  
            images_input.append(img) 
            
        if n_APs == 3 and simulation == 'cells':
            maps_p = os.path.join(dir_maps, name_maps)
            maps_p = np.asarray(io.imread(maps_p))/255
            img = np.stack([image_buildings, image_Tx1, image_Tx2, image_Tx3, maps_p], axis = 2)  
            images_input.append(img)
            
        if n_APs == 4 and simulation == 'maps':
            img = np.stack([image_buildings, image_Tx1, image_Tx2, image_Tx3, image_Tx4], axis = 2)  
            images_input.append(img) 
            
        if n_APs == 4 and simulation == 'cells':
            maps_p = os.path.join(dir_maps, name_maps)
            maps_p = np.asarray(io.imread(maps_p))/255
            img = np.stack([image_buildings, image_Tx1, image_Tx2, image_Tx3, image_Tx4, maps_p], axis = 2)  
            images_input.append(img)
            
        if n_APs == 5 and simulation == 'maps':
            img = np.stack([image_buildings, image_Tx1, image_Tx2, image_Tx3, image_Tx4, image_Tx5], axis = 2)  
            images_input.append(img) 
            
        if n_APs == 5 and simulation == 'cells':
            maps_p = os.path.join(dir_maps, name_maps)
            maps_p = np.asarray(io.imread(maps_p))/255
            img = np.stack([image_buildings, image_Tx1, image_Tx2, image_Tx3, image_Tx4, image_Tx5, maps_p], axis = 2)  
            images_input.append(img)

        if simulation == 'maps':    
            img_name_out = os.path.join(dir_maps, name_maps)  
            image_out = np.asarray(io.imread(img_name_out))/255
            images_out.append(image_out)
        
        if simulation == 'cells' and n_APs != 1:    
            img_name_out = os.path.join(dir_cells, name_maps)  
            image_out = np.asarray(io.imread(img_name_out))
            
            if not np.array_equal(np.unique(image_out), np.array([0, 85, 170, 255])):
                print(img_name_out)
                print(np.unique(image_out))
                print("---------------------------------------")
            images_out.append(image_out)
            
    # if simulation == 'cells':
    #     images_input = np.expand_dims(images_input, axis=3)        

    images_input = np.array(images_input)
    images_out = np.array(images_out)

    if simulation == 'maps': 
        images_out = np.expand_dims(images_out, axis=3)
        # Normalize out    
        # images_out = normalize_map(images_out, 0)
        
    if simulation == 'cells': 
        # Normalize maps in input
        # images_input = normalize_map(images_input, -1)
        
        labelencoder = LabelEncoder()
        n, h, w = images_out.shape
        images_out_reshaped = images_out.reshape(-1,1)
        images_out_reshaped_encoded = labelencoder.fit_transform(images_out_reshaped)
        images_out_encoded_original_shape = images_out_reshaped_encoded.reshape(n, h, w)
        images_out = np.expand_dims(images_out_encoded_original_shape, axis=3)
        
        # class_weights = class_weight.compute_class_weight('balanced',
        #                                                 np.unique(images_out_reshaped_encoded),
        #                                                 images_out_reshaped_encoded)
        # print("Class weights are:", class_weights)
    
    return images_input, images_out

In [ ]:
print(f"We can choice {scenarios * positions} dates")

In [ ]:
maps_inds = np.arange(1, (scenarios*positions) + 1, 1, dtype = np.int16) 
np.random.seed(42) 
np.random.shuffle(maps_inds)
x_train, y_train = data(maps_inds, simulation = simulation, n_APs = n_APs, 
                        dir_dataset_txs = dir_dataset_txs, dir_cells = dir_cells, 
                        dir_maps = dir_maps, dir_dataset_esc = dir_dataset_esc, 
                        positions = positions, ini = 0, end = 7999) # 133 distint positions
                        # positions = positions, ini = 0, end = 200)

In [ ]:
scenarios_test = 20
positions_test = 10
maps_inds_test = np.arange(1, (scenarios_test*positions_test) + 1, 1, dtype = np.int16) 
np.random.seed(42) 
np.random.shuffle(maps_inds_test)
x_test, y_test = data(maps_inds_test, simulation = simulation, n_APs = n_APs, 
                        dir_dataset_txs = dir_dataset_txs, dir_cells = dir_cells, 
                        dir_maps = dir_maps, dir_dataset_esc = dir_dataset_esc, 
                        positions = positions_test, phase = 'test')
                        # positions = positions, ini = 50, end = 70)

In [ ]:
# Save data for load after
# np.savez(dir_dataset + 'Singles cells/Data/' + simulation + '_' + str(n_APs) +'_data_train.npz', x_train = x_train, y_train = y_train)
# np.savez(dir_dataset + 'Singles cells/Data/' + simulation + '_' + str(n_APs) +'_data_test.npz', x_test = x_test, y_test = y_test)

In [ ]:
# Load data saved before
# data_load = np.load(dir_dataset + 'Singles cells/Data/' + simulation + '_' + str(n_APs) +'_data_train.npz')
# x_train = data_load['x_train']
# y_train = data_load['y_train']

# data_load = np.load(dir_dataset + 'Singles cells/Data/' + simulation + '_' + str(n_APs) +'_data_test.npz')
# x_test = data_load['x_test']
# y_test = data_load['y_test']

Dimenstions data

In [ ]:
print(x_train.shape)
print(y_train.shape)

In [ ]:
print(x_test.shape)
print(y_test.shape)

#### Data split train (validation) and test

In [ ]:
# x_train, x_test, y_train, y_test = train_test_split(images_input, images_out, 
#                                                   test_size = porc_test, random_state = 0)

#### Some data visualization

In [ ]:
grapics = 5
imgs_in = x_train[:grapics,...]
imgs_o = y_train[:grapics,...]
def plot_data(imgs_in, imgs_o):
    plt.figure(figsize=(40,40))
    for i in range(grapics):
        plt.subplot(4, 5, i+1)
        inp = imgs_in[i,...]
        o = imgs_o[i, ...]
        di = 0
        if simulation == 'maps':
            color_map = cm.get_cmap('jet') 
            img_colory = color_map(o)
        else:
            img_colory = o

        # plt.imshow(inp[:,:,di])
        plt.imshow(img_colory[:,:,0])
        # print(img_colory[:,:,0])
        # print(np.unique(inp[:,:,di]))
        print(np.unique(img_colory[:,:,0]))
        
        plt.axis('Off')
    plt.tight_layout()
    plt.show()

plot_data(imgs_in, imgs_o)

#### Output preparation of cells in one-hot vectors (if is requered) and training model

Some functions for metrics manuals

In [ ]:
def pixelwise_error(y_true, y_pred):
    return K.mean(K.cast(K.not_equal(K.argmax(y_true, axis=-1), K.argmax(y_pred, axis=-1)), K.floatx()), axis=[1,2])

def mean_iou(y_true, y_pred):
    num_classes = K.int_shape(y_pred)[-1]
    iou = 0
    for c in range(num_classes):
        y_true_c = K.cast(K.equal(y_true[...,c], 1), 'float32')
        y_pred_c = K.cast(K.equal(K.argmax(y_pred, axis=-1), c), 'float32')
        intersection = K.sum(y_true_c * y_pred_c, axis=[1, 2])
        union = K.sum(K.maximum(y_true_c, y_pred_c), axis=[1, 2])
        iou_c = K.mean((intersection + K.epsilon()) / (union + K.epsilon()))
        iou += iou_c
    return iou / num_classes

In [ ]:
# Parameters for cross validation
n_splits=10
kf = KFold(n_splits = n_splits) 

# Lists metrics
if simulation == 'maps':
    scores_loss = []
    scores_rmse = []
    scores_mae = []

if simulation == 'cells':
    scores_loss = []
    scores_acc = []
    scores_pe = []
    scores_iou = []

hist_arr = []

In [ ]:
lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate = learning_rate,
    decay_steps = 30000,
    decay_rate = 0.30)

optimizer = optimizers.Adam(learning_rate = lr_schedule)

In [ ]:
if simulation == 'cells':
    model.compile(optimizer = optimizer, loss='categorical_crossentropy', metrics=['categorical_accuracy', pixelwise_error, mean_iou])
    
if simulation == 'maps':
    model.compile(optimizer = optimizer, loss='mean_squared_error', metrics=[tf.keras.metrics.RootMeanSquaredError(), 
                                                                             tf.keras.metrics.MeanAbsoluteError()])


In [ ]:
def training(dir_dataset, model, n_splits, simulation = '', n_APs = None, x_train = [], y_train = [], 
             epochs = None, batch_size = None): # Se usa validación cruzada
    
    callbacks = [keras.callbacks.ModelCheckpoint(dir_dataset + '/Models/' + str(n_APs) + 'AP/' +
                                                    simulation + '_' + str(n_APs) + '_model.h5', 
                                                    save_best_only = True)]
    
    s_loss =  1e6
    
    fold_no = 1

    if simulation == 'cells':
        print('uniques values of classes: ', np.unique(y_train))         
        
        y_train_cat = to_categorical(y_train, num_classes = n_APs)

        for train_index, val_index in kf.split(x_train):

            print(f'Fold: {fold_no}')
            X_train_cv, X_val_cv = x_train[train_index], x_train[val_index]
            y_train_cv, y_val_cv = y_train_cat[train_index], y_train_cat[val_index]

            hist = model.fit(X_train_cv, y_train_cv, 
                        batch_size = batch_size, 
                        verbose = 1, 
                        epochs = epochs//n_splits, 
                        validation_data = (X_val_cv, y_val_cv), 
                        callbacks = callbacks,
                        shuffle = False,
                        #class_weight = class_weights
                        )
            
            hist_arr.append(hist)

            # Model test
            score = model.evaluate(X_val_cv, y_val_cv, verbose=0)
            print(f'Score for fold {fold_no}: {model.metrics_names[0]} of {score[0]}; {model.metrics_names[1]} of {score[1]}; {model.metrics_names[2]} of {score[2]}; {model.metrics_names[3]} of {score[3]}')
            
            if s_loss > score[0]:
                s_loss = score[0]
                print(f"The best metrics fold: {fold_no} --- loss: {score[0]}; acc {score[1]}; pe: {score[2]}; iou_ {score[3]}")
            
            scores_loss.append(score[0])
            scores_acc.append(score[1])
            scores_pe.append(score[2])
            scores_iou.append(score[3])
            fold_no += 1
            
            X_train_cv = None
            X_val_cv = None
            y_train_cv = None
            y_val_cv = None
            train_index = None
            val_index = None
            
        # Average results cross validation
        print(f'Loss: {np.mean(scores_loss)}; acc: {np.mean(scores_acc)}; pe: {np.mean(scores_pe)}; iou: {np.mean(scores_iou)}')
        
        return scores_loss, scores_acc, scores_pe, scores_iou, hist_arr
        
    if simulation == 'maps':
        
        for train_index, val_index in kf.split(x_train):

            print(f'Fold: {fold_no}')
            X_train_cv, X_val_cv = x_train[train_index], x_train[val_index]
            y_train_cv, y_val_cv = y_train[train_index], y_train[val_index]

            hist = model.fit(X_train_cv, y_train_cv, 
                        batch_size = batch_size, 
                        verbose = 1, 
                        epochs = epochs//n_splits, 
                        validation_data = (X_val_cv, y_val_cv), 
                        callbacks = callbacks,
                        shuffle = False)
            
            hist_arr.append(hist)

            # Model test
            score = model.evaluate(X_val_cv, y_val_cv, verbose=0)
            print(f'Score for fold {fold_no}: {model.metrics_names[0]} of {score[0]}; {model.metrics_names[1]} of {score[1]}; {model.metrics_names[2]} of {score[2]}')
            
            if s_loss > score[0]:
                s_loss = score[0]
                print(f"The best metrics fold: {fold_no} --- loss: {score[0]}; rmse {score[1]}; mae: {score[2]}")
            
            scores_loss.append(score[0])
            scores_rmse.append(score[1])
            scores_mae.append(score[2])
            fold_no += 1
            
            X_train_cv = None
            X_val_cv = None
            y_train_cv = None
            y_val_cv = None
            train_index = None
            val_index = None

        # Average results cross validation
        print(f'Loss: {np.mean(scores_loss)}; rmse: {np.mean(scores_rmse)}; mae: {np.mean(scores_mae)}')
    
        return scores_loss, scores_rmse, scores_mae, hist_arr
        
    ###############################################################################    
    
if simulation == 'maps':
    scores_loss, scores_rmse, scores_mae, hist_arr = training(dir_dataset = dir_dataset, model = model, n_splits = n_splits, simulation = simulation, n_APs = n_APs, 
                                                            x_train = x_train, y_train = y_train, epochs = epochs, batch_size = batch_size)
    
if simulation == 'cells':
    scores_loss, scores_acc, scores_pe, scores_iou, hist_arr = training(dir_dataset = dir_dataset, model = model, n_splits = n_splits, simulation = simulation, n_APs = n_APs, 
                                                                        x_train = x_train, y_train = y_train, epochs = epochs, batch_size = batch_size)

In [ ]:
# Save data metrcis
if simulation == 'cells':
    np.save(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + simulation + '_' + str(n_APs) +'_metrics_loss.npy', np.array(scores_loss))
    np.save(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + simulation + '_' + str(n_APs) +'_metrics_acc.npy', np.array(scores_acc))
    np.save(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + simulation + '_' + str(n_APs) +'_metrics_pe.npy', np.array(scores_pe))
    np.save(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + simulation + '_' + str(n_APs) +'_metrics_iou.npy', np.array(scores_iou))

if simulation == 'maps':
    np.save(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + simulation + '_' + str(n_APs) +'_metrics_loss.npy', np.array(scores_loss))
    np.save(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + simulation + '_' + str(n_APs) +'_metrics_rmse.npy', np.array(scores_rmse))
    np.save(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + simulation + '_' + str(n_APs) +'_metrics_mae.npy', np.array(scores_mae))


#### Data predictions

In [ ]:
# Load the best model saved
# model = load_model(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + simulation + '_' + str(n_APs) + '_model.h5')

In [ ]:
model.load_weights(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + simulation + '_' + str(n_APs) + '_model.h5')

In [ ]:
def scale_matrix(matrix):
    min_val = np.min(matrix)
    max_val = np.max(matrix)
    scaled_matrix = ((matrix - min_val) / (max_val - min_val)) * 255
    return scaled_matrix.astype(int)


In [ ]:
# plain = np.asarray(io.imread('E:/DataSet5GHz/Models/Test others/im2_bin.png'))/255
# tx1 = np.asarray(io.imread('E:/DataSet5GHz/Models/Test others/tx1_im2.png'))/255
# # tx2 = np.asarray(io.imread('E:/Test_V/tx2_im1.png'))/255
# # out_in = np.asarray(io.imread('E:/Test_V/engineering_map_1_1.png'))/255
# in1 = np.stack([plain, tx1], axis = 2) 

# out = np.asarray(io.imread('E:/DataSet5GHz/Models/Test others/engineering_map_2_1.png'))/255
# out = np.expand_dims(out, -1)
# test_img = in1
# ground_truth = out
# test_img_input = np.expand_dims(test_img, 0)
# inicio = time.time()
# prediction = model.predict(test_img_input)
# fin = time.time()
# print(f'Time used was: {fin-inicio} seconds') 

# print(f"Shape 1 image test: {test_img.shape}")
# print(f"Shape 2 image test: {test_img_input.shape}")
# print(f"Shape image prediction: {prediction.shape}")
# print(f"Shape image true: {ground_truth.shape}")

# # Show output images
# plt.figure(figsize=(6, 6))
# # plt.subplot(231)
# if simulation == 'cells':
#     print(r'Testing Label')
# if simulation == 'maps':
#     print(r'Testing Level')
# color_map = cm.get_cmap('jet') 
# if simulation == 'cells':
#     m1 = color_map(scale_matrix(ground_truth[:,:,0]))
# else:
#     m1 = color_map(ground_truth[:,:,0])
# # m1 = ground_truth[:,:,0]
# m1 = color_map(ground_truth[:,:,0])
# plt.imshow(m1)
# plt.axis('Off')
# plt.imsave('E:/DataSet5GHz/Models/Test others/true_2_.png', m1) 
# plt.show()

# # plt.subplot(232)
# plt.figure(figsize=(6,6))

# print(r'Prediction on test image')
# if simulation == 'cells':
#     predicted_img = np.argmax(prediction, axis=3)[0,:,:]
#     color_map = cm.get_cmap('jet') 
#     if simulation == 'cells':
#         m2 = color_map(scale_matrix(predicted_img))
#     else:
#         m2 = color_map(predicted_img)
#     # plt.imshow(m2)
#     # m2 = predicted_img
#     plt.imshow(m2)
#     plt.axis('Off')    
# if simulation == 'maps':
#     m2 = color_map(prediction[0,:,:,0])
#     plt.imshow(m2)
#     plt.axis('Off')
# plt.imsave('E:/DataSet5GHz/Models/Test others/predict_2_.png', m2)   
# plt.show()

# plt.figure(figsize=(12, 8))

# if simulation == 'maps':
#     if n_APs == 1:
#         plt.subplot(231)
#         plt.title(r'Testing image plain')
#         plt.imshow(test_img[:,:,0])
#         plt.axis('Off')

#         plt.subplot(232)
#         plt.title(r'Testing image AP 1')
#         plt.imshow(test_img[:,:,1])
#         plt.axis('Off')

#     if n_APs == 2:
#         plt.subplot(231)
#         plt.title(r'TestingiImage plain')
#         plt.imshow(test_img[:,:,0])
#         plt.axis('Off')

#         plt.subplot(232)
#         plt.title(r'Testing image AP 1')
#         plt.imshow(test_img[:,:,1])
#         plt.axis('Off')
        
#         plt.subplot(233)
#         plt.title(r'Testing image AP 2')
#         plt.imshow(test_img[:,:,2])
#         plt.axis('Off')

#     if n_APs >= 3:            
#         plt.subplot(231)
#         plt.title(r'TestingiImage plain')
#         plt.imshow(test_img[:,:,0])        
#         plt.axis('Off')

#         plt.subplot(232)
#         plt.title(r'Testing image AP 1')
#         plt.imshow(test_img[:,:,1])
#         plt.axis('Off')
        
#         plt.subplot(233)
#         plt.title(r'Testing image AP 2')
#         plt.imshow(test_img[:,:,2])
#         plt.axis('Off')

#         plt.subplot(234)
#         plt.title(r'Testing image AP 3')
#         plt.imshow(test_img[:,:,3])
#         plt.axis('Off')
    
# if simulation == 'cells':
#     if n_APs == 2:
#         plt.subplot(231)
#         plt.title(r'TestingiImage plain')
#         plt.imshow(test_img[:,:,0])        
#         plt.axis('Off')

#         plt.subplot(232)
#         plt.title(r'Testing image AP 1')
#         plt.imshow(test_img[:,:,1])
#         plt.axis('Off')
        
#         plt.subplot(233)
#         plt.title(r'Testing image AP 2')
#         plt.imshow(test_img[:,:,2])
#         plt.axis('Off')

#         plt.figure(figsize=(12, 8))
#         # plt.subplot(234)
#         # plt.title(r'Testing image power')
#         m3 = color_map(test_img[:,:,-1])
#         plt.imshow(m3)
#         plt.axis('Off')
#         plt.imsave(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + str(count) + '_' + simulation + '_' + str(n_APs) + '_cell_map.png', m3) 

#     if n_APs >= 3:            
#         plt.subplot(231)
#         plt.title(r'TestingiImage plain')
#         plt.imshow(test_img[:,:,0])
#         plt.axis('Off')

#         plt.subplot(232)
#         plt.title(r'Testing image AP 1')
#         plt.imshow(test_img[:,:,1])
#         plt.axis('Off')
        
#         plt.subplot(233)
#         plt.title(r'Testing image AP 2')
#         plt.imshow(test_img[:,:,2])
#         plt.axis('Off')

#         plt.subplot(234)
#         plt.title(r'Testing image AP 3')
#         plt.imshow(test_img[:,:,3])
#         plt.axis('Off')

#         plt.figure(figsize=(12, 8))
#         # plt.subplot(235)
#         # plt.title(r'Testing image power')
#         m3 = color_map(test_img[:,:,-1])
#         plt.imshow(m3)
#         plt.axis('Off')
#         plt.imsave(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + str(count) + '_' + simulation + '_' + str(n_APs) + '_cell_map.png', m3) 

In [ ]:
count = 1
while count <= 1:
    test_img_number = random.randint(0, len(x_test)-1)
    test_img = x_test[test_img_number]
    ground_truth = y_test[test_img_number]
    test_img_input = np.expand_dims(test_img, 0)
    inicio = time.time()
    prediction = model.predict(test_img_input)
    fin = time.time()
    print(f'Time used was: {fin-inicio} seconds') 

    print(f"Shape 1 image test: {test_img.shape}")
    print(f"Shape 2 image test: {test_img_input.shape}")
    print(f"Shape image prediction: {prediction.shape}")
    print(f"Shape image true: {ground_truth.shape}")

    # Show output images
    plt.figure(figsize=(6, 6))
    # plt.subplot(231)
    if simulation == 'cells':
        print(r'Testing Label')
    if simulation == 'maps':
        print(r'Testing Level')
    color_map = cm.get_cmap('jet') 
    if simulation == 'cells':
        m1 = color_map(scale_matrix(ground_truth[:,:,0]))
    else:
        m1 = color_map(ground_truth[:,:,0])
    # m1 = ground_truth[:,:,0]
    plt.imshow(m1)
    plt.axis('Off')
    plt.imsave(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + str(count) + '_' + simulation + '_' + str(n_APs) + '_true.png', m1) 
    plt.show()

    # plt.subplot(232)
    plt.figure(figsize=(6,6))

    print(r'Prediction on test image')
    if simulation == 'cells':
        predicted_img = np.argmax(prediction, axis=3)[0,:,:]
        color_map = cm.get_cmap('jet') 
        if simulation == 'cells':
            m2 = color_map(scale_matrix(predicted_img))
        else:
            m2 = color_map(predicted_img)
        # plt.imshow(m2)
        # m2 = predicted_img
        plt.imshow(m2)
        plt.axis('Off')    
    if simulation == 'maps':
        m2 = color_map(prediction[0,:,:,0])
        plt.imshow(m2)
        plt.axis('Off')
    plt.imsave(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + str(count) + '_' + simulation + '_' + str(n_APs) + '_eval.png', m2)   
    plt.show()

    plt.figure(figsize=(12, 8))

    if simulation == 'maps':
        if n_APs == 1:
            plt.subplot(231)
            plt.title(r'Testing image plain')
            plt.imshow(test_img[:,:,0])
            plt.axis('Off')

            plt.subplot(232)
            plt.title(r'Testing image AP 1')
            plt.imshow(test_img[:,:,1])
            plt.axis('Off')

        if n_APs == 2:
            plt.subplot(231)
            plt.title(r'TestingiImage plain')
            plt.imshow(test_img[:,:,0])
            plt.axis('Off')

            plt.subplot(232)
            plt.title(r'Testing image AP 1')
            plt.imshow(test_img[:,:,1])
            plt.axis('Off')
            
            plt.subplot(233)
            plt.title(r'Testing image AP 2')
            plt.imshow(test_img[:,:,2])
            plt.axis('Off')

        if n_APs >= 3:            
            plt.subplot(231)
            plt.title(r'TestingiImage plain')
            plt.imshow(test_img[:,:,0])
            plt.axis('Off')

            plt.subplot(232)
            plt.title(r'Testing image AP 1')
            plt.imshow(test_img[:,:,1])
            plt.axis('Off')
            
            plt.subplot(233)
            plt.title(r'Testing image AP 2')
            plt.imshow(test_img[:,:,2])
            plt.axis('Off')

            plt.subplot(234)
            plt.title(r'Testing image AP 3')
            plt.imshow(test_img[:,:,3])
            plt.axis('Off')
        
    if simulation == 'cells':
        if n_APs == 2:
            plt.subplot(231)
            plt.title(r'TestingiImage plain')
            plt.imshow(test_img[:,:,0])
            plt.axis('Off')

            plt.subplot(232)
            plt.title(r'Testing image AP 1')
            plt.imshow(test_img[:,:,1])
            plt.axis('Off')
            
            plt.subplot(233)
            plt.title(r'Testing image AP 2')
            plt.imshow(test_img[:,:,2])
            plt.axis('Off')

            plt.figure(figsize=(12, 8))
            # plt.subplot(234)
            # plt.title(r'Testing image power')
            m3 = color_map(test_img[:,:,-1])
            plt.imshow(m3)
            plt.axis('Off')
            plt.imsave(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + str(count) + '_' + simulation + '_' + str(n_APs) + '_cell_map.png', m3) 

        if n_APs >= 3:            
            plt.subplot(231)
            plt.title(r'TestingiImage plain')
            plt.imshow(test_img[:,:,0])
            plt.axis('Off')

            plt.subplot(232)
            plt.title(r'Testing image AP 1')
            plt.imshow(test_img[:,:,1])
            plt.axis('Off')
            
            plt.subplot(233)
            plt.title(r'Testing image AP 2')
            plt.imshow(test_img[:,:,2])
            plt.axis('Off')

            plt.subplot(234)
            plt.title(r'Testing image AP 3')
            plt.imshow(test_img[:,:,3])
            plt.axis('Off')

            plt.figure(figsize=(12, 8))
            # plt.subplot(235)
            # plt.title(r'Testing image power')
            m3 = color_map(test_img[:,:,-1])
            plt.imshow(m3)
            plt.axis('Off')
            plt.imsave(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + str(count) + '_' + simulation + '_' + str(n_APs) + '_cell_map.png', m3) 

    count = count + 1


#### Data Evaluation on model

some metrics

In [ ]:
if simulation == 'cells':
    model.compile(optimizer = optimizer, loss='categorical_crossentropy', metrics=['categorical_accuracy', pixelwise_error, mean_iou])
    # Model test
    score = model.evaluate(x_test, to_categorical(y_test, num_classes = n_APs), verbose=0)
    print(f'Score for data test: {model.metrics_names[0]} of {np.round(score[0], 2)}; {model.metrics_names[1]} of {np.round(score[1], 2)}; {model.metrics_names[2]} of {np.round(score[2], 2)}; {model.metrics_names[3]} of {np.round(score[3], 2)}')
            
if simulation == 'maps':
    model.compile(optimizer = optimizer, loss='mean_squared_error', metrics=[tf.keras.metrics.RootMeanSquaredError(), 
                                                                             tf.keras.metrics.MeanAbsoluteError()])
    score = model.evaluate(x_test, y_test, verbose=0)
    print(f'Score for data test: {model.metrics_names[0]} of {score[0]}; {model.metrics_names[1]} of {score[1]}; {model.metrics_names[2]} of {score[2]}')

Confusion matrix

In [ ]:
font = FontProperties(family='Times New Roman', size=12)

y_pred = model.predict(x_test)
confusion_mtx = confusion_matrix(np.ravel(y_test), np.ravel(np.argmax(y_pred, axis=3)))

# Obtener el número total de píxeles
total_pixels = np.sum(confusion_mtx)

# Redondear los valores de la matriz de confusión
confusion_mtx_rounded = np.round(confusion_mtx, decimals=4)

# Calcular la matriz de confusión en porcentajes
confusion_mtx_percentage = confusion_mtx_rounded / total_pixels

min_value = 0
max_value = n_APs
vector = [str(i) for i in range(min_value, max_value)]

# Mostrar la matriz de confusión en porcentajes
cm_display = ConfusionMatrixDisplay(confusion_mtx_percentage, display_labels=vector)

fig, ax = plt.subplots(figsize=(8, 6))
cm_display.plot(cmap=plt.cm.Blues, ax=ax)
plt.xlabel("Predicted", fontproperties=font)  # Aplicar la fuente
plt.ylabel("True", fontproperties=font)  # Aplicar la fuente

# Crear un objeto PdfPages para guardar el archivo PDF
pdf_pages = PdfPages('E:/DataSet5GHz/Models/Test others/' + str(n_APs) + '_matriz_confusion.pdf')
pdf_pages.savefig(fig)  # Guardar la figura en el archivo PDF
pdf_pages.close()  # Cerrar el archivo PDF

plt.show()


### Visualization and save history metrics

In [ ]:
def extract_metrics_maps(history):    
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    rmse = history.history['root_mean_squared_error']
    val_rmse = history.history['val_root_mean_squared_error']
    mae = history.history['mean_absolute_error']
    val_mae = history.history['val_mean_absolute_error']
    return loss, val_loss, rmse, val_rmse, mae, val_mae

def extract_metrics_cells(history):
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    acc = history.history['categorical_accuracy']
    val_acc = history.history['val_categorical_accuracy']
    pixel_e = history.history['pixelwise_error']
    val_pixel_e = history.history['val_pixelwise_error']
    iou = history.history['mean_iou']
    val_iou = history.history['val_mean_iou']
    return loss, val_loss, acc, val_acc, pixel_e, val_pixel_e, iou, val_iou


In [ ]:
losses = []
val_losses = []

rmse_scores = []
val_rmse_scores = []
mae_scores = []
val_mae_scores = []

acc_scores = []
val_acc_scores = []
pixel_e_scores = []
val_pixel_e_scores = []
iou_scores = []
val_iou_scores = []


In [ ]:
if simulation == 'maps':
    for history in hist_arr:
        loss, val_loss, rmse, val_rmse, mae, val_mae = extract_metrics_maps(history)
        losses.append(loss)
        val_losses.append(val_loss)
        rmse_scores.append(rmse)
        val_rmse_scores.append(val_rmse)
        mae_scores.append(mae)
        val_mae_scores.append(val_mae)
        
if simulation == 'cells':
    for history in hist_arr:
        loss, val_loss, acc, val_acc, pixel_e, val_pixel_e, iou, val_iou = extract_metrics_cells(history)
        losses.append(loss)
        val_losses.append(val_loss)
        acc_scores.append(acc)
        val_acc_scores.append(val_acc)
        pixel_e_scores.append(pixel_e)
        val_pixel_e_scores.append(val_pixel_e)
        iou_scores.append(iou)
        val_iou_scores.append(val_iou)
        


Save history metrics

In [ ]:
if simulation == 'maps':
        np.savez(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + simulation + '_' + str(n_APs) +'_history.npz', 
                losses = losses,
                val_losses = val_losses,
                rmse_scores = rmse_scores,
                val_rmse_scores = val_rmse_scores,
                mae_scores = mae_scores,
                val_mae_scores = val_mae_scores)
        
if simulation == 'cells':
        np.savez(dir_dataset + '/Models/' + str(n_APs) + 'AP/' + simulation + '_' + str(n_APs) +'_history.npz', 
                losses = losses,
                val_losses = val_losses,
                acc_scores = acc_scores,
                val_acc_scores = val_acc_scores,
                pixel_e_scores = pixel_e_scores,
                val_pixel_e_scores = val_pixel_e_scores,
                iou_scores = iou_scores,
                val_iou_scores = val_iou_scores)
        